In [1]:
import torch
import torch.nn as nn
from copy import deepcopy

# ══════════════════════════════════════════════
#  1. OPTIMIZERS FROM SCRATCH
# ══════════════════════════════════════════════

class BatchGD:
    """
    Batch Gradient Descent — uses the ENTIRE dataset every step.
        θ = θ - lr * ∇L(θ; X_all, y_all)
    """
    def __init__(self, params, lr: float = 0.01):
        self.params = list(params)
        self.lr = lr

    def step(self):
        with torch.no_grad():
            for p in self.params:
                if p.grad is not None:
                    p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


class MiniBatchGD:
    """
    Mini-Batch Gradient Descent — identical update rule to BatchGD,
    but the training loop feeds mini-batches instead of the full dataset.
        θ = θ - lr * ∇L(θ; X_batch, y_batch)
    (The difference is in the training loop, not the optimizer.)
    """
    def __init__(self, params, lr: float = 0.01):
        self.params = list(params)
        self.lr = lr

    def step(self):
        with torch.no_grad():
            for p in self.params:
                if p.grad is not None:
                    p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


class SGDScratch:
    """
    Stochastic Gradient Descent — same update rule, batch_size=1.
        θ = θ - lr * ∇L(θ; x_i, y_i)
    (Again, the difference is in the training loop.)
    """
    def __init__(self, params, lr: float = 0.01):
        self.params = list(params)
        self.lr = lr

    def step(self):
        with torch.no_grad():
            for p in self.params:
                if p.grad is not None:
                    p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


class MomentumSGD:
    """
    SGD with Momentum — accumulates a velocity to accelerate through
    flat regions and dampen oscillations.

        v = β * v + (1 - β) * ∇L      ← exponential moving average of gradients
        θ = θ - lr * v
    """
    def __init__(self, params, lr: float = 0.01, beta: float = 0.9):
        self.params = list(params)
        self.lr = lr
        self.beta = beta
        self.v = [torch.zeros_like(p) for p in self.params]   # velocity

    def step(self):
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is not None:
                    self.v[i] = self.beta * self.v[i] + (1 - self.beta) * p.grad
                    p -= self.lr * self.v[i]

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


class RMSPropScratch:
    """
    RMSProp — adapts the learning rate per-parameter by dividing
    by the running average of squared gradients.

        s = β * s + (1 - β) * (∇L)²   ← tracks gradient magnitude
        θ = θ - lr * ∇L / (√s + ε)    ← large past grads → smaller step
    """
    def __init__(self, params, lr: float = 0.001, beta: float = 0.99, eps: float = 1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta = beta
        self.eps = eps
        self.s = [torch.zeros_like(p) for p in self.params]   # squared grad cache

    def step(self):
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is not None:
                    self.s[i] = self.beta * self.s[i] + (1 - self.beta) * p.grad ** 2
                    p -= self.lr * p.grad / (self.s[i].sqrt() + self.eps)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


class AdamScratch:
    """
    Adam — combines Momentum (1st moment) + RMSProp (2nd moment)
    with BIAS CORRECTION to fix early-step underestimates.

        m = β₁ * m + (1 - β₁) * ∇L           ← 1st moment (mean)
        v = β₂ * v + (1 - β₂) * (∇L)²        ← 2nd moment (variance)

        m̂ = m / (1 - β₁ᵗ)                     ← bias-corrected mean
        v̂ = v / (1 - β₂ᵗ)                     ← bias-corrected variance

        θ = θ - lr * m̂ / (√v̂ + ε)
    """
    def __init__(self, params, lr: float = 0.001,
                 beta1: float = 0.9, beta2: float = 0.999, eps: float = 1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = [torch.zeros_like(p) for p in self.params]   # 1st moment
        self.v = [torch.zeros_like(p) for p in self.params]   # 2nd moment
        self.t = 0                                             # timestep

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is not None:
                    # Update biased moments
                    self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * p.grad
                    self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * p.grad ** 2

                    # Bias correction
                    m_hat = self.m[i] / (1 - self.beta1 ** self.t)
                    v_hat = self.v[i] / (1 - self.beta2 ** self.t)

                    # Update
                    p -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


# ══════════════════════════════════════════════
#  2. TRAINING HELPERS
# ══════════════════════════════════════════════

def make_dataset(n: int = 500, seed: int = 42):
    """Synthetic binary classification dataset."""
    torch.manual_seed(seed)
    X = torch.randn(n, 4)
    w_true = torch.tensor([1.5, -2.0, 0.5, 1.0])
    logits = X @ w_true + 0.3
    y = (torch.sigmoid(logits) > 0.5).float().unsqueeze(1)
    return X, y


def make_model(seed: int = 0):
    """Small network: 4 → 16 → 8 → 1."""
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Linear(4, 16), nn.ReLU(),
        nn.Linear(16, 8), nn.ReLU(),
        nn.Linear(8, 1),  nn.Sigmoid(),
    )


def get_batches(X, y, batch_size: int):
    """Yield mini-batches."""
    n = len(X)
    idx = torch.randperm(n)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        sel = idx[start:end]
        yield X[sel], y[sel]


# ══════════════════════════════════════════════
#  3. UNIFIED TRAINING FUNCTION
# ══════════════════════════════════════════════

def train(model, optimizer, X, y, epochs: int = 100,
          batch_size: int = None, label: str = "") -> list:
    """
    Train a model.
      batch_size = None      → Batch GD (full dataset)
      batch_size = 1         → SGD (one sample)
      batch_size = 32, etc.  → Mini-batch GD
    """
    criterion = nn.BCELoss()
    losses = []

    if batch_size is None:
        batch_size = len(X)   # full batch

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        n_batches = 0

        for X_b, y_b in get_batches(X, y, batch_size):
            pred = model(X_b)
            loss = criterion(pred, y_b)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)

    return losses


# ══════════════════════════════════════════════
#  4. EXAMPLE USAGE — compare all optimizers
# ══════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 65)
    print("  Optimizers from Scratch — Full Comparison")
    print("=" * 65)

    X, y = make_dataset(500)
    EPOCHS = 100
    SEED = 42

    # ── 4a. Gradient Descent Variants (same optimizer, different batch sizes) ──
    print("\n── Part 1: GD Variants (lr=0.1) ──\n")
    print(f"  {'Method':<22} {'Batch':>6}   Loss@25   Loss@50   Loss@100  Acc")
    print(f"  {'─'*22} {'─'*6}   {'─'*8}  {'─'*8}  {'─'*8}  {'─'*5}")

    gd_configs = [
        ("Batch GD",       BatchGD,      None, 0.1),
        ("Mini-batch GD",  MiniBatchGD,  32,   0.1),
        ("SGD (batch=1)",  SGDScratch,   1,    0.1),
    ]

    for name, OptimizerClass, bs, lr in gd_configs:
        model = make_model(SEED)
        opt = OptimizerClass(model.parameters(), lr=lr)
        losses = train(model, opt, X, y, epochs=EPOCHS, batch_size=bs)

        with torch.no_grad():
            acc = ((model(X) >= 0.5).float() == y).float().mean().item()

        bs_str = str(bs) if bs else "all"
        print(f"  {name:<22} {bs_str:>6}   {losses[24]:.4f}    {losses[49]:.4f}    "
              f"{losses[99]:.4f}    {acc:.0%}")

    # ── 4b. Advanced Optimizers ──
    print(f"\n── Part 2: Advanced Optimizers (mini-batch=32) ──\n")
    print(f"  {'Optimizer':<22} {'LR':>8}   Loss@25   Loss@50   Loss@100  Acc")
    print(f"  {'─'*22} {'─'*8}   {'─'*8}  {'─'*8}  {'─'*8}  {'─'*5}")

    advanced_configs = [
        ("Vanilla SGD",    SGDScratch,   0.1),
        ("Momentum (β=.9)",MomentumSGD,  0.1),
        ("RMSProp",        RMSPropScratch, 0.001),
        ("Adam (full)",    AdamScratch,  0.001),
    ]

    for name, OptimizerClass, lr in advanced_configs:
        model = make_model(SEED)
        opt = OptimizerClass(model.parameters(), lr=lr)
        losses = train(model, opt, X, y, epochs=EPOCHS, batch_size=32)

        with torch.no_grad():
            acc = ((model(X) >= 0.5).float() == y).float().mean().item()

        print(f"  {name:<22} {lr:>8}   {losses[24]:.4f}    {losses[49]:.4f}    "
              f"{losses[99]:.4f}    {acc:.0%}")

    # ── 4c. Verify Adam matches PyTorch ──
    print(f"\n── Part 3: Verify Adam vs PyTorch Adam ──\n")

    torch.manual_seed(SEED)
    model_ours = make_model(SEED)
    opt_ours = AdamScratch(model_ours.parameters(), lr=0.001)

    torch.manual_seed(SEED)
    model_pt = make_model(SEED)
    opt_pt = torch.optim.Adam(model_pt.parameters(), lr=0.001)

    criterion = nn.BCELoss()
    for epoch in range(50):
        # Ours
        pred = model_ours(X)
        loss_ours = criterion(pred, y)
        opt_ours.zero_grad()
        loss_ours.backward()
        opt_ours.step()

        # PyTorch
        pred = model_pt(X)
        loss_pt = criterion(pred, y)
        opt_pt.zero_grad()
        loss_pt.backward()
        opt_pt.step()

    print(f"  After 50 full-batch steps:")
    print(f"    Our Adam loss:     {loss_ours.item():.6f}")
    print(f"    PyTorch Adam loss: {loss_pt.item():.6f}")

    # Compare parameters
    max_diff = max(
        (p1.data - p2.data).abs().max().item()
        for p1, p2 in zip(model_ours.parameters(), model_pt.parameters())
    )
    print(f"    Max param diff:    {max_diff:.2e}")
    print(f"    Match: {'✓' if max_diff < 1e-5 else '✗  (small float differences expected)'}")

    # ── 4d. Show bias correction effect ──
    print(f"\n── Part 4: Adam bias correction effect ──\n")

    model_bc   = make_model(SEED)
    model_nobc = make_model(SEED)

    opt_bc   = AdamScratch(model_bc.parameters(), lr=0.001)

    # Adam WITHOUT bias correction
    class AdamNoBiasCorrection(AdamScratch):
        def step(self):
            self.t += 1
            with torch.no_grad():
                for i, p in enumerate(self.params):
                    if p.grad is not None:
                        self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * p.grad
                        self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * p.grad ** 2
                        # NO bias correction — use raw m and v
                        p -= self.lr * self.m[i] / (self.v[i].sqrt() + self.eps)

    opt_nobc = AdamNoBiasCorrection(model_nobc.parameters(), lr=0.001)

    losses_bc   = train(model_bc,   opt_bc,   X, y, epochs=EPOCHS, batch_size=32)
    losses_nobc = train(model_nobc, opt_nobc, X, y, epochs=EPOCHS, batch_size=32)

    print(f"  {'Epoch':<8} {'With correction':>16} {'Without':>16}")
    print(f"  {'─'*8} {'─'*16} {'─'*16}")
    for e in [1, 5, 10, 25, 50, 100]:
        print(f"  {e:<8} {losses_bc[e-1]:>16.4f} {losses_nobc[e-1]:>16.4f}")

    print("\n  → Bias correction helps most in early steps (when m,v are")
    print("    still near zero and would underestimate the true moments).")

    # ── Summary ──
    print(f"""
══════════════════════════════════════════════════════════════
  OPTIMIZER CHEAT SHEET
══════════════════════════════════════════════════════════════

  Batch GD        θ -= lr * ∇L(full dataset)
                  ✓ Stable        ✗ Slow, no escaping minima

  Mini-batch GD   θ -= lr * ∇L(batch)
                  ✓ Fast + stable ✗ Need to tune batch size

  SGD (batch=1)   θ -= lr * ∇L(single sample)
                  ✓ Noisy → escapes minima   ✗ Very noisy

  Momentum        v = β·v + (1-β)·∇L;  θ -= lr·v
                  ✓ Accelerates through flat regions

  RMSProp         s = β·s + (1-β)·(∇L)²;  θ -= lr·∇L/√s
                  ✓ Per-param adaptive learning rate

  Adam            m,v (momentum + RMSProp) + bias correction
                  ✓ Best default choice for most problems
══════════════════════════════════════════════════════════════
""")

  Optimizers from Scratch — Full Comparison

── Part 1: GD Variants (lr=0.1) ──

  Method                  Batch   Loss@25   Loss@50   Loss@100  Acc
  ────────────────────── ──────   ────────  ────────  ────────  ─────
  Batch GD                  all   0.6783    0.6141    0.3874    90%
  Mini-batch GD              32   0.0423    0.0203    0.0097    100%
  SGD (batch=1)               1   0.0011    0.0002    0.0000    100%

── Part 2: Advanced Optimizers (mini-batch=32) ──

  Optimizer                    LR   Loss@25   Loss@50   Loss@100  Acc
  ────────────────────── ────────   ────────  ────────  ────────  ─────
  Vanilla SGD                 0.1   0.0423    0.0203    0.0097    100%
  Momentum (β=.9)             0.1   0.0426    0.0194    0.0097    100%
  RMSProp                   0.001   0.0903    0.0329    0.0102    100%
  Adam (full)               0.001   0.1165    0.0408    0.0155    100%

── Part 3: Verify Adam vs PyTorch Adam ──

  After 50 full-batch steps:
    Our Adam loss:     0